# 3.2 Managing Messages (adaptado a lab local)

**Qué se ve aquí:** dos middlewares para controlar el historial que llega al modelo:
- `SummarizationMiddleware`: cuando el historial pasa de un umbral (`trigger`), resume lo viejo con un modelo y conserva lo último (`keep`).
- `@before_agent` + `RemoveMessage`: borra mensajes del state (aquí, todos los `ToolMessage`) antes de que corra el agente.

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

## Summarize messages

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago. El resumen usaba "gpt-4o-mini" (pago) → también MODEL.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import SummarizationMiddleware
#
# agent = create_agent(
#     model="gpt-5-nano",
#     checkpointer=InMemorySaver(),
#     middleware=[
#         SummarizationMiddleware(
#             model="gpt-4o-mini",
#             trigger=("tokens", 100),
#             keep=("messages", 1)
#         )
#     ],
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago. El resumen usaba "gpt-4o-mini" (pago) → también MODEL.
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=MODEL,   # el mismo modelo ya cargado en Ollama: cambiar de modelo en CPU cuesta segundos de carga,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

In [ ]:
# Sin cambios
print(response["messages"][0].content)

## Trim/delete messages

In [ ]:
# Sin cambios
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# agent = create_agent(
#     model="gpt-5-nano",
#     checkpointer=InMemorySaver(),
#     middleware=[trim_messages],
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
agent = create_agent(
    model=MODEL,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
# Sin cambios
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

In [ ]:
# Sin cambios
print(response["messages"][-1].content)